[Reference](https://christianbernecker.medium.com/feature-engineering-for-machine-learning-a-practical-guide-with-scikit-learn-43e2d3a79b01)

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

# Create the sample raw data
data = {
    'tenure_months': [3, 24, 1, 15, 8],
    'monthly_charge': [9.99, 19.99, 19.99, 14.99, 9.99],
    'last_login_date': ['2025-09-15', '2025-09-10', '2025-08-20', '2025-09-18', '2025-07-01'],
    'subscription_plan': ['Basic', 'Premium', 'Premium', 'Basic', 'Basic']
}
df = pd.DataFrame(data)
print("Original Raw Data:")
print(df)

Original Raw Data:
   tenure_months  monthly_charge last_login_date subscription_plan
0              3            9.99      2025-09-15             Basic
1             24           19.99      2025-09-10           Premium
2              1           19.99      2025-08-20           Premium
3             15           14.99      2025-09-18             Basic
4              8            9.99      2025-07-01             Basic


# 1. Handling Dates: From a Timestamp to a Signal

In [2]:
# Assume 'today' is September 19th, 2025
today = datetime.strptime('2025-09-19', '%Y-%m-%d')
# Convert to datetime and calculate days since last login
df['last_login_date'] = pd.to_datetime(df['last_login_date'])
df['days_since_last_login'] = (today - df['last_login_date']).dt.days
print("\nData after engineering 'days_since_last_login':")
print(df[['last_login_date', 'days_since_last_login']].head())


Data after engineering 'days_since_last_login':
  last_login_date  days_since_last_login
0      2025-09-15                      4
1      2025-09-10                      9
2      2025-08-20                     30
3      2025-09-18                      1
4      2025-07-01                     80


# 2. Binning Numbers: From Raw Counts to Meaningful Groups

In [3]:
from sklearn.preprocessing import KBinsDiscretizer

# We need to reshape the data for the transformer
tenure_data = df[['tenure_months']]
# Initialize the binner to create 3 groups
binner = KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform', subsample=None)
# Fit and transform the data
df['tenure_group'] = binner.fit_transform(tenure_data)
print("\nData after binning 'tenure_months':")
print(df[['tenure_months', 'tenure_group']].head())


Data after binning 'tenure_months':
   tenure_months  tenure_group
0              3           0.0
1             24           2.0
2              1           0.0
3             15           1.0
4              8           0.0


# 3. Creating Interaction Features: Combining Clues

In [4]:
from sklearn.preprocessing import PolynomialFeatures

# Select the features to interact
interaction_features = df[['tenure_months', 'monthly_charge']]
# Create interaction terms (e.g., tenure * monthly_charge)
poly = PolynomialFeatures(interaction_only=True, include_bias=False)
interactions = poly.fit_transform(interaction_features)
# Add the new feature to the DataFrame
df['tenure_charge_interaction'] = interactions[:, -1] # The last column is the interaction term
print("\nData after creating interaction feature:")
print(df[['tenure_months', 'monthly_charge', 'tenure_charge_interaction']].head())


Data after creating interaction feature:
   tenure_months  monthly_charge  tenure_charge_interaction
0              3            9.99                      29.97
1             24           19.99                     479.76
2              1           19.99                      19.99
3             15           14.99                     224.85
4              8            9.99                      79.92


# 4. Encoding Categories: Making Text Understandable

In [5]:
from sklearn.preprocessing import OneHotEncoder
# We need to reshape the data for the transformer
plan_data = df[['subscription_plan']]
# Initialize the encoder
encoder = OneHotEncoder(sparse_output=False)
# Fit and transform, then create a new DataFrame with the encoded columns
encoded_plans = encoder.fit_transform(plan_data)
encoded_df = pd.DataFrame(encoded_plans, columns=encoder.get_feature_names_out(['subscription_plan']))
# Join the new encoded columns back to the original DataFrame
df = df.join(encoded_df)
print("\nData after One-Hot Encoding 'subscription_plan':")
print(df[['subscription_plan_Basic', 'subscription_plan_Premium']])


Data after One-Hot Encoding 'subscription_plan':
   subscription_plan_Basic  subscription_plan_Premium
0                      1.0                        0.0
1                      0.0                        1.0
2                      0.0                        1.0
3                      1.0                        0.0
4                      1.0                        0.0


In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Redefine the original DataFrame to start fresh
df_original = pd.DataFrame(data)
# Convert date column to datetime first, as ColumnTransformer works on the data
df_original['last_login_date'] = pd.to_datetime(df_original['last_login_date'])
df_original['days_since_last_login'] = (today - df_original['last_login_date']).dt.days

# Define the transformers for each column type
# 1. Binner for tenure
# 2. One-Hot Encoder for subscription plan
preprocessor = ColumnTransformer(
    transformers=[
        ('binner', KBinsDiscretizer(n_bins=3, encode='ordinal', strategy='uniform', subsample=None), ['tenure_months']),
        ('onehot', OneHotEncoder(sparse_output=False), ['subscription_plan'])
    ],
    remainder='passthrough' # Keep other columns (like monthly_charge, days_since_last_login)
)
# Create a pipeline that applies the preprocessing
pipeline = Pipeline(steps=[('preprocessor', preprocessor)])
# Apply the entire pipeline to the data
transformed_data = pipeline.fit_transform(df_original)
print("\n--- Final Transformed Data from Pipeline ---")
# The output is a NumPy array, ready for a model!
print(transformed_data)


--- Final Transformed Data from Pipeline ---
[[0.0 1.0 0.0 9.99 Timestamp('2025-09-15 00:00:00') 4]
 [2.0 0.0 1.0 19.99 Timestamp('2025-09-10 00:00:00') 9]
 [0.0 0.0 1.0 19.99 Timestamp('2025-08-20 00:00:00') 30]
 [1.0 1.0 0.0 14.99 Timestamp('2025-09-18 00:00:00') 1]
 [0.0 1.0 0.0 9.99 Timestamp('2025-07-01 00:00:00') 80]]
